# 🧪 Document Classification Pipeline Analysis

This notebook provides a tool for detailed inspection of the classification graph. You can:
1. **Skip OCR** by providing text directly.
2. **Analyze Logprobs** and token distribution for each node.
3. **Debug Decision Margins** and HITL triggers.

In [17]:
%load_ext autoreload
%autoreload 2

import os
import json
import asyncio
from pathlib import Path
from dotenv import load_dotenv

# Force load .env from root
load_dotenv("../.env") 

from config.settings import AppConfig
from graph.builder import build_classification_graph
from graph.state import create_initial_state

print("✅ Modules loaded successfully")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Modules loaded successfully


In [18]:
# 1. Initialize Config and Graph
config = AppConfig()
graph = build_classification_graph(config, use_checkpointer=False)

print(f"📌 Current Model: {'OpenAI (' + config.openai_model + ')' if config.openai_api_key else 'Azure OpenAI'}")
print(f"📌 Margin Threshold: {config.margin_threshold}")

📌 Current Model: OpenAI (gpt-4o-mini)
📌 Margin Threshold: 1.5


### 📊 Analysis Helper
This function prints the logprob distribution captured at each node.

In [19]:
def print_node_analysis(node_name, logprobs, margin, conf, threshold):
    if not logprobs:
        print(f"\n❌ No logprob data found for {node_name}")
        return
    
    # Helper to handle both Pydantic objects (.attr) and dictionary (['key']) access
    def get_v(obj, key):
        if isinstance(obj, dict): return obj.get(key)
        return getattr(obj, key, None)

    top1_t = get_v(logprobs, "top1_token")
    top1_l = get_v(logprobs, "top1_logprob")
    top1_p = get_v(logprobs, "top1_prob_pct")
    top2_t = get_v(logprobs, "top2_token")
    top2_l = get_v(logprobs, "top2_logprob")
    top2_p = get_v(logprobs, "top2_prob_pct")
    
    # Format strings safely
    top1_l_str = f"{top1_l:.4f}" if top1_l is not None else "N/A"
    top1_p_str = f"{top1_p:.2f}%" if top1_p is not None else "N/A"
    top2_l_str = f"{top2_l:.4f}" if top2_l is not None else "N/A"
    top2_p_str = f"{top2_p:.2f}%" if top2_p is not None else "N/A"
    conf_str = f"{conf:.2f}%" if conf is not None else "0.00%"
    margin_str = f"{margin:.4f}" if margin is not None else "0.0000"

    print(f"\n--- {node_name.upper()} ANALYSIS ---")
    print(f"Winning Token: {top1_t} (Logprob: {top1_l_str}, Prob: {top1_p_str})")
    print(f"Runner-up:     {top2_t} (Logprob: {top2_l_str}, Prob: {top2_p_str})")
    
    status = "✅ SAFE" if (margin or 0) >= threshold else "⚠️ UNCERTAIN (HITL)"
    print(f"Margin Score:  {margin_str} (Threshold: {threshold}) -> {status}")
    print(f"Final Confidence Score: {conf_str}")

async def run_pipeline(text=None, file_path="dummy.pdf"):
    state = create_initial_state(
        document_id="notebook_test",
        file_path=file_path,
        file_type="pdf"
    )
    
    if text:
        state["azure_ocr_text"] = text
        print("🚀 Running with provided text (skipping OCR)...\n")
    else:
        print(f"🚀 Running with file: {file_path}...\n")
        
    final_state = await graph.ainvoke(state)
    
    # 1. Root Stage
    print_node_analysis(
        "root", 
        final_state.get("root_logprobs"), 
        final_state.get("root_margin"), 
        final_state.get("root_confidence_pct"), 
        config.margin_threshold
    )
    
    # 2. Sub Stage
    if final_state.get("sub_code"):
        print_node_analysis(
            "specialist", 
            final_state.get("sub_logprobs"), 
            final_state.get("sub_margin"), 
            final_state.get("sub_confidence_pct"), 
            config.margin_threshold
        )
        print(f"\nFinal Result: {final_state['root_code']} -> {final_state['sub_code']}")
    else:
        print(f"\nFinal Result: {final_state.get('root_code')} (Terminated early)")
        
    print(f"Trail: {' -> '.join(final_state.get('execution_trail', []))}")
    return final_state

### 🧪 Define input text

In [ ]:
test_text = """
12/02/2022 14:58:35 โรง พยาบาล พระราม เก้า PRARAM 9 HOSPITAL DOCTOR 'S ORDER SHEET Name :
นาง XXXX XXX DOB : 5 ก.พ . 2506 Allergy : No Blood Reaction : Age : 59 Years Page HN : XXXXX
วัน ที่ บันทึก : Date Orders for 1 day Only Hour Hour HOMEMED 14 ก.ค . 65 17:27 • [ ยา เดิม ]CRAvill 0.5%
EYE DROPS 5 ml.(levoFLOXacin)( ลี โว ฟ ลอก ซา ซิน ) รุ่ INSTILL INTO EYE(S) 1 DROP BID (MORNING &
AT BE - Qty : |1 Charge Amt. 14 ก.ค . 65 17:27 - [ ยา เดิม ]LEVOPRONT 30 mg/5 ml syr 60
mL(levodropropizine). ORAL 5 ml (MILLILITRE) BIDPC - - Qty :| 1 Charge Amt. . 14 ก.ค . 65 17:27 - [ ยา
เดิม ]NASONEX 60 50 mcg NASAL SPRAY 60 DOSES(mometasone). SPRAY INTO NOSTRILS 2 PUFF QID -
Qty : 1 Charge Amt. 14 ก.ค . 65 17:27 - [ ยา เดิม ]RAPIHALER SYMBICORT 160+4.5 mcg(120 (DOSES)
(budesonide+formoterol). สุด ยา เข้า ทาง ปาก 2 DOSE BID (MORNING & AT BE - 14 ก.ค . 65 17:27 - Qty 1
Charge Amt. - AUGMENTIN 1 gm BID TAB( ออก เมน ติน ) ORAL 1 TABLET BIOPC ยา ฆ่า เชื้อ !/ รับ ประทาน
ติดต่อ ตาม แพทย์ สั่ง จน ครบ - Qty : 6 tbe Charge Amt. 366 - ILIADIN 0.05% NASAL SPRAY 10
mL(oxymetazoline) 14 ก.ค . 65 17:27 (>/=6 yo). SPRAY INTO NOSTRIL 1 PUFF QID - Qty !: 1 026 Charge
Amt. 185 14 ก.ค . 65 17:27 • PSEUDOEPHEDRINE 60 mg TAB^P.2. ORAL 1 TABLET TIDPC - Qty : 15 เม็ด
Charge Amt. 210 14 ก.ค . 65 17:27 - ROTUSS/ROpect 10+100 mg CAP(codeine;guaifenesin)^N3. 1 ORAL
1 TABLET TID (AFTER BREAKFAST - Qty: 15 win Charge Amt. 240 หมายเหตุ ห้าม ใช้ ตัว ย่อ ดัง ต่อ ไป นี้
1)U 2)Q.D 3)Q.O.D, AD, 222 4) MS 5)MSO4 6)MgSO4 เด้ง คลอด บ๊อบ /Y A mai 21 ให้ เขียน "0" นำ หน้า
จุดทศนิยม (0. X mg) To be taken-home Discharge PRARAN OSPITAL สำเนา ถูก ต้อง ORDER No. DRI6507-
05611 Date Orders for continuation
"""

result = await run_pipeline(text=test_text)

🚀 Running with provided text (skipping OCR)...

2026-04-26 17:59:56 [info     ] graph_ocr_ingestion_start      document_id=notebook_test
2026-04-26 17:59:56 [info     ] graph_ocr_ingestion_skipped    document_id=notebook_test reason=text_provided
2026-04-26 17:59:56 [info     ] graph_root_router_start        document_id=notebook_test
2026-04-26 17:59:58 [info     ] graph_root_router_complete     confidence_pct=100.0 document_id=notebook_test is_uncertain=False margin=17.875 root_code=MED
2026-04-26 17:59:58 [info     ] graph_med_specialist_start     document_id=notebook_test
2026-04-26 17:59:59 [info     ] graph_med_specialist_complete  confidence_pct=100.0 document_id=notebook_test hospital_name=None is_uncertain=True margin=100.0 sub_code=OTH

--- ROOT ANALYSIS ---
Winning Token: MED (Logprob: 0.0000, Prob: 100.00%)
Runner-up:     MED (Logprob: -17.8750, Prob: 0.00%)
Margin Score:  17.8750 (Threshold: 1.5) -> ✅ SAFE
Final Confidence Score: 100.00%

--- SPECIALIST ANALYSIS ---
Winning

### 🧪 Scenario 2: Ambiguous Document (Low Margin)
Try a document that looks like both or neither.

In [21]:
ambiguous_text = """
This is a letter about a claim. It mentions a doctor but looks like an invoice.
Please check the amount of $500 for medical consultation.
"""

result = await run_pipeline(text=ambiguous_text)

🚀 Running with provided text (skipping OCR)...

2026-04-26 18:00:18 [info     ] graph_ocr_ingestion_start      document_id=notebook_test
2026-04-26 18:00:18 [info     ] graph_ocr_ingestion_skipped    document_id=notebook_test reason=text_provided
2026-04-26 18:00:18 [info     ] graph_root_router_start        document_id=notebook_test
2026-04-26 18:00:18 [info     ] graph_root_router_complete     confidence_pct=100.0 document_id=notebook_test is_uncertain=False margin=10.25 root_code=MED
2026-04-26 18:00:18 [info     ] graph_med_specialist_start     document_id=notebook_test
2026-04-26 18:00:19 [info     ] graph_med_specialist_complete  confidence_pct=100.0 document_id=notebook_test hospital_name=None is_uncertain=True margin=100.0 sub_code=OTH

--- ROOT ANALYSIS ---
Winning Token: MED (Logprob: -0.0000, Prob: 100.00%)
Runner-up:     NON (Logprob: -10.2500, Prob: 0.00%)
Margin Score:  10.2500 (Threshold: 1.5) -> ✅ SAFE
Final Confidence Score: 100.00%

--- SPECIALIST ANALYSIS ---
Winning